In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

print(f"✅ Spark session created: {spark.version}")

26/03/09 17:26:59 WARN Utils: Your hostname, odinsbeard-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/09 17:27:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 17:27:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 17:27:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Spark session created: 3.5.0


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Load ALL data
df_sales = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/sales_*.csv")

df_products = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/products_*.csv")

df_users = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/users_*.csv")

# Register views
df_sales.createOrReplaceTempView("sales")
df_products.createOrReplaceTempView("products")
df_users.createOrReplaceTempView("users")

print(f"\n📊 Data loaded:")
print(f"   • Sales: {df_sales.count():,} rows")
print(f"   • Products: {df_products.count():,} rows")
print(f"   • Users: {df_users.count():,} rows")


📊 Data loaded:


   • Sales: 20,000,000 rows
   • Products: 400,000 rows


[Stage 12:================================================>         (5 + 1) / 6]

   • Users: 4,000,000 rows


In [9]:
# Quick checks
print("Sales schema:")
df_sales.printSchema()
print("\nSample sales data:")
df_sales.show(10, truncate=False)

print("Products schema:")
df_products.printSchema()
print("\nSample product data:")
df_products.show(10, truncate=False)

print("Users schema:")
df_users.printSchema()
print("\nSample users data:")
df_users.show(10, truncate=False)

Sales schema:
root
 |-- sale_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- final_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)


Sample sales data:
+-------+-------+----------+--------+------+----------------+------------+----------+--------------+---------+
|sale_id|user_id|product_id|quantity|price |discount_percent|final_amount|sale_date |payment_method|country  |
+-------+-------+----------+--------+------+----------------+------------+----------+--------------+---------+
|1      |744523 |74347     |1       |221.24|0               |221.24      |2025-04-14|PayPal        |Brazil   |
|2      |280736 |39152     |4       |286.94|5               |1090.37     |2024-03-20|Bank Transfer

In [10]:
# Regular GROUP BY (collapses rows)
print("📌 GROUP BY - collapses details:")
df_sales.groupBy("country") \
    .agg(sum("final_amount").alias("total")) \
    .show(10)

# Window Function (keeps all rows + adds calculation)
print("\n📌 WINDOW FUNCTION - keeps details + adds running total:")
window_spec = Window.partitionBy("country").orderBy("sale_date")

df_sales.select("country", "sale_date", "final_amount",
    sum("final_amount").over(window_spec).alias("running_total")) \
    .show(10)

📌 GROUP BY - collapses details:


+---------+--------------------+
|  country|               total|
+---------+--------------------+
|  Germany|1.1486470551599944E9|
|   France| 1.147935600049995E9|
|      USA|1.1475363208499959E9|
|       UK|1.1485043907800007E9|
|   Canada|1.1488513600499916E9|
|   Brazil|1.1481623346599889E9|
|    Japan|1.1474787411999967E9|
|Australia|1.1458380956599884E9|
+---------+--------------------+


📌 WINDOW FUNCTION - keeps details + adds running total:


[Stage 29:>                                                         (0 + 1) / 1]

+-------+----------+------------+------------------+
|country| sale_date|final_amount|     running_total|
+-------+----------+------------+------------------+
|Germany|2023-01-01|      752.27|1029044.3699999995|
|Germany|2023-01-01|      154.47|1029044.3699999995|
|Germany|2023-01-01|      619.68|1029044.3699999995|
|Germany|2023-01-01|      596.68|1029044.3699999995|
|Germany|2023-01-01|      450.47|1029044.3699999995|
|Germany|2023-01-01|      294.12|1029044.3699999995|
|Germany|2023-01-01|      832.35|1029044.3699999995|
|Germany|2023-01-01|      425.08|1029044.3699999995|
|Germany|2023-01-01|      497.01|1029044.3699999995|
|Germany|2023-01-01|      875.58|1029044.3699999995|
+-------+----------+------------+------------------+
only showing top 10 rows



In [11]:
# Find top products by revenue
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number

# Calculate product revenue
product_revenue = df_sales.groupBy("product_id") \
    .agg(sum("final_amount").alias("revenue"))

# Join with product names
product_performance = product_revenue.join(df_products, "product_id") \
    .select("product_name", "category", "revenue")

# Define window for ranking within category
window_spec = Window.partitionBy("category").orderBy(col("revenue").desc())

# Add different ranks
ranked_products = product_performance \
    .withColumn("rank", rank().over(window_spec)) \
    .withColumn("dense_rank", dense_rank().over(window_spec)) \
    .withColumn("row_num", row_number().over(window_spec))

print("📊 Top 3 products per category:")
ranked_products.filter(col("rank") <= 3) \
    .orderBy("category", "rank") \
    .show(20, truncate=False)


📊 Top 3 products per category:


[Stage 34:>                                                         (0 + 6) / 6]

+---------------+-----------+-----------------+----+----------+-------+
|product_name   |category   |revenue          |rank|dense_rank|row_num|
+---------------+-----------+-----------------+----+----------+-------+
|Magazine Plus  |Books      |77767.96999999999|1   |1         |1      |
|Cookbook Lite  |Books      |74422.99         |2   |2         |2      |
|Fiction Max    |Books      |74091.45000000001|3   |3         |3      |
|Dress Plus     |Clothing   |74523.43000000001|1   |1         |1      |
|Socks Basic    |Clothing   |74308.88         |2   |2         |2      |
|Shorts Pro     |Clothing   |74246.84000000001|3   |3         |3      |
|Headphones Max |Electronics|75724.17         |1   |1         |1      |
|Mouse Plus     |Electronics|74313.68         |2   |2         |2      |
|Smartphone Plus|Electronics|73055.75         |3   |3         |3      |
|Tools Pro      |Home       |75769.23999999999|1   |1         |1      |
|Tools Max      |Home       |75025.66999999998|2   |2         |2

In [12]:
# Daily sales trend with running totals
daily_sales = df_sales.groupBy("sale_date") \
    .agg(sum("final_amount").alias("daily_revenue")) \
    .orderBy("sale_date")

# Define window for running total
window_spec = Window.orderBy("sale_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Define window for 7-day moving average
window_spec_7day = Window.orderBy("sale_date") \
    .rowsBetween(-6, Window.currentRow)

daily_analysis = daily_sales \
    .withColumn("running_total", sum("daily_revenue").over(window_spec)) \
    .withColumn("moving_avg_7day", avg("daily_revenue").over(window_spec_7day))

print("📈 Daily sales with running total and 7-day average:")
daily_analysis.show(20)

📈 Daily sales with running total and 7-day average:


26/03/09 19:13:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:13:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:13:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:13:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 1

+----------+-----------------+--------------------+------------------+
| sale_date|    daily_revenue|       running_total|   moving_avg_7day|
+----------+-----------------+--------------------+------------------+
|2023-01-01|8344250.389999996|   8344250.389999996| 8344250.389999996|
|2023-01-02|8494198.619999994| 1.683844900999999E7| 8419224.504999995|
|2023-01-03|8329097.010000002|2.5167546019999992E7| 8389182.006666664|
|2023-01-04|       8447347.07| 3.361489308999999E7| 8403723.272499997|
|2023-01-05|8325709.419999999| 4.194060250999999E7| 8388120.501999998|
|2023-01-06|8251309.149999999| 5.019191165999999E7|8365318.6099999985|
|2023-01-07|8486690.569999998| 5.867860222999999E7|  8382657.46142857|
|2023-01-08|8430134.309999997|6.7108736539999984E7| 8394926.592857141|
|2023-01-09|8316905.500000002| 7.542564203999999E7| 8369599.004285714|
|2023-01-10|8458165.069999997| 8.388380710999998E7| 8388037.298571427|
|2023-01-11|8220504.190000001| 9.210431129999998E7| 8355631.172857142|
|2023-

In [13]:
# Segment customers into spending quartiles
customer_totals = df_sales.groupBy("user_id") \
    .agg(sum("final_amount").alias("total_spent"))

# Define window for ntile
window_spec = Window.orderBy(col("total_spent").desc())

customer_segments = customer_totals \
    .withColumn("quartile", ntile(4).over(window_spec)) \
    .withColumn("decile", ntile(10).over(window_spec)) \
    .withColumn("percentile_rank", percent_rank().over(window_spec))

print("📊 Customer spending segments:")
customer_segments.groupBy("quartile") \
    .agg(count("*").alias("customers"),
         min("total_spent").alias("min_spent"),
         max("total_spent").alias("max_spent"),
         avg("total_spent").alias("avg_spent")) \
    .orderBy("quartile") \
    .show()

📊 Customer spending segments:


26/03/09 19:17:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:17:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:17:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:17:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:17:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:17:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:17:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:17:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatc

+--------+---------+---------+------------------+------------------+
|quartile|customers|min_spent|         max_spent|         avg_spent|
+--------+---------+---------+------------------+------------------+
|       1|   913711|  3670.62|17409.940000000002|  5467.74514431789|
|       2|   913710|  1967.96|           3670.62| 2741.646573092221|
|       3|   913710|   876.36|           1967.96|1391.9071623162629|
|       4|   913710|     8.06|            876.36| 448.8794379726701|
+--------+---------+---------+------------------+------------------+



In [14]:
# Segment customers into spending quartiles
customer_totals = df_sales.groupBy("user_id") \
    .agg(sum("final_amount").alias("total_spent"))

# Define window for ntile
window_spec = Window.orderBy(col("total_spent").desc())

customer_segments = customer_totals \
    .withColumn("quartile", ntile(4).over(window_spec)) \
    .withColumn("decile", ntile(10).over(window_spec)) \
    .withColumn("percentile_rank", percent_rank().over(window_spec))

print("📊 Customer spending segments:")
customer_segments.groupBy("quartile") \
    .agg(count("*").alias("customers"),
         min("total_spent").alias("min_spent"),
         max("total_spent").alias("max_spent"),
         avg("total_spent").alias("avg_spent")) \
    .orderBy("quartile") \
    .show()

📊 Customer spending segments:


26/03/09 19:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:20:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 19:20:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:20:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:20:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:20:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:20:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:20:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatc

+--------+---------+---------+------------------+------------------+
|quartile|customers|min_spent|         max_spent|         avg_spent|
+--------+---------+---------+------------------+------------------+
|       1|   913711|  3670.62|17409.940000000002|  5467.74514431789|
|       2|   913710|  1967.96|           3670.62| 2741.646573092221|
|       3|   913710|   876.36|           1967.96|1391.9071623162629|
|       4|   913710|     8.06|            876.36| 448.8794379726701|
+--------+---------+---------+------------------+------------------+



In [15]:
# Find each customer's favorite product category
# Step 1: Get spending by customer and category
customer_category = df_sales.join(df_products, "product_id") \
    .groupBy("user_id", "category") \
    .agg(sum("final_amount").alias("category_spent"))

# Step 2: Rank categories for each customer
window_spec = Window.partitionBy("user_id") \
    .orderBy(col("category_spent").desc())

customer_preferences = customer_category \
    .withColumn("rank", rank().over(window_spec)) \
    .withColumn("total_customer_spend", 
                sum("category_spent").over(Window.partitionBy("user_id"))) \
    .withColumn("category_pct", 
                round(col("category_spent") / col("total_customer_spend") * 100, 2))

# Step 3: Get top category for each customer
top_category = customer_preferences.filter(col("rank") == 1) \
    .join(df_users.select("user_id", "first_name", "last_name", "country"), "user_id") \
    .select("user_id", "first_name", "last_name", "country", 
            "category", "category_spent", "category_pct")

print("🎯 Each customer's favorite category:")
top_category.show(20, truncate=False)

🎯 Each customer's favorite category:


26/03/09 19:24:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:24:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:24:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:24:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:24:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:24:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:24:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:24:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 19:24:13 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+----------+---------+---------+-----------+------------------+------------+
|user_id|first_name|last_name|country  |category   |category_spent    |category_pct|
+-------+----------+---------+---------+-----------+------------------+------------+
|12     |Michelle  |Taylor   |USA      |Electronics|667.9200000000001 |54.45       |
|22     |Anthony   |Brown    |USA      |Clothing   |2892.75           |42.95       |
|26     |Kenneth   |Moore    |Brazil   |Books      |3746.21           |82.45       |
|27     |Barbara   |Miller   |UK       |Books      |1817.42           |34.12       |
|28     |Donald    |Gonzalez |Canada   |Sports     |1035.34           |45.65       |
|31     |Charles   |Ramirez  |UK       |Books      |2801.06           |52.51       |
|34     |Elizabeth |Sanchez  |France   |Electronics|2816.12           |32.22       |
|44     |Donald    |Lopez    |UK       |Clothing   |1212.23           |36.62       |
|53     |Jennifer  |Williams |Germany  |Clothing   |1234.14      

In [16]:
# Just put SQL inside the string!
spark.sql("""
    SELECT 
        country,
        COUNT(*) as order_count,
        SUM(final_amount) as total_revenue
    FROM sales
    GROUP BY country
    ORDER BY total_revenue DESC
""").show()

[Stage 72:===================================================>    (11 + 1) / 12]

+---------+-----------+--------------------+
|  country|order_count|       total_revenue|
+---------+-----------+--------------------+
|   Canada|    2501886|1.1488513600499916E9|
|  Germany|    2500462|1.1486470551599944E9|
|       UK|    2501002|1.1485043907800007E9|
|   Brazil|    2501094|1.1481623346599889E9|
|   France|    2500495| 1.147935600049995E9|
|      USA|    2498441|1.1475363208499959E9|
|    Japan|    2498267|1.1474787411999967E9|
|Australia|    2498353|1.1458380956599884E9|
+---------+-----------+--------------------+



In [21]:
# Enable SQL magic - run this once
spark.sql("SELECT 1").show()

+---+
|  1|
+---+
|  1|
+---+



In [23]:
spark.sql("""

WITH customer_stats AS (
    SELECT 
        u.user_id,
        u.first_name || ' ' || u.last_name as customer_name,
        u.country,
        COUNT(*) as purchase_count,
        SUM(s.final_amount) as total_spent
    FROM sales s
    JOIN users u ON s.user_id = u.user_id
    GROUP BY u.user_id, u.first_name, u.last_name, u.country
)
SELECT 
    country,
    COUNT(*) as customer_count,
    AVG(total_spent) as avg_spent_per_customer,
    SUM(total_spent) as total_revenue
FROM customer_stats
GROUP BY country
ORDER BY total_revenue DESC

""").show()

26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 21:03:34 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+--------------+----------------------+--------------------+
|  country|customer_count|avg_spent_per_customer|       total_revenue|
+---------+--------------+----------------------+--------------------+
|  Germany|        457572|    2517.8672564754884| 1.152105556280002E9|
|   Brazil|        456401|    2517.7833400671575|1.1491188341899908E9|
|   Canada|        457030|     2513.143892479697| 1.148582153179996E9|
|Australia|        456793|    2512.3406399616333|1.1476196179499943E9|
|       UK|        456994|    2510.9930453135053|     1.14750875575E9|
|      USA|        456813|     2511.700264747278|1.1473773330399983E9|
|    Japan|        456868|    2507.6014058546675|1.1456428390900102E9|
|   France|        456370|     2508.926548480399|1.1449988089299998E9|
+---------+--------------+----------------------+--------------------+



In [1]:
spark.stop()

NameError: name 'spark' is not defined